# Qwen moderation — RL-style evaluation & improvement

This notebook ties the **OpenEnv reward** (heuristic graders in `env/graders/`) to your **Qwen** policy loaded via `inference.py`.

**What is actually happening?**
- **Before**: greedy decoding (`temperature=0`) — same path as Discord/meme moderation.
- **After (default here)**: *test-time* **best-of-N** — sample several JSON actions, keep the one with **highest grader reward**. This improves accuracy at the cost of extra forward passes (no weight update).
- **Optional weight update**: run `python scripts/rl_trainer.py lora-sft` (needs `pip install -r requirements-train.txt` + GPU recommended), set `HF_ADAPTER_PATH` in `.env`, then re-run **Eval slot → after** with greedy decoding.

Metrics are written to `results/rl_training_metrics.json` and shown on the frontend **RL training** page.

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "inference.py").exists():
    ROOT = ROOT.parent  # if launched from notebooks/
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("ROOT =", ROOT)

In [ ]:
# Load Qwen (first cell may download weights)
import inference  # noqa: F401
from scripts.rl_trainer import (
    evaluate_policy,
    load_posts,
    write_metrics_bundle,
    merge_slot,
    build_iterations,
)

MODEL_ID = getattr(inference, "HF_MODEL_ID", None) or getattr(inference, "MODEL_NAME", "?")
print("Model:", MODEL_ID)

In [ ]:
# --- Configure eval ---
DIFFICULTIES = ["easy", "medium", "hard"]  # subset of task_difficulty in data/posts.json
LIMIT = None  # e.g. 40 for a quick run; None = all posts
BEST_OF = 4   # test-time improvement for the "after" curve

posts = load_posts(DIFFICULTIES, LIMIT)
print(len(posts), "posts")

In [ ]:
import inference as inf

before = evaluate_policy(
    posts,
    lambda p: inf._call_llm(p),
    label="greedy_baseline",
    best_of=1,
)
before["model_id"] = MODEL_ID
before

In [ ]:
after = evaluate_policy(
    posts,
    lambda p: inf._call_llm(p),
    label=f"best_of_{BEST_OF}",
    best_of=BEST_OF,
    temperature=0.65,
)
after["model_id"] = MODEL_ID
after

In [ ]:
OUT = ROOT / "results" / "rl_training_metrics.json"
iterations = build_iterations(before, after)
write_metrics_bundle(
    OUT,
    before=before,
    after=after,
    model_id=MODEL_ID,
    notes=(
        f"Notebook RLtrainer: greedy vs best-of-{BEST_OF}. "
        "For LoRA gains run `python scripts/rl_trainer.py lora-sft` then "
        "`scripts/rl_trainer.py eval-slot after --greedy`."
    ),
    iterations=iterations,
)
print("Wrote", OUT)

## Optional: update only the **after** slot (e.g. after LoRA)
```python
after2 = evaluate_policy(posts, lambda p: inf._call_llm(p), label="post_lora_greedy", best_of=1)
after2["model_id"] = MODEL_ID
merge_slot(OUT, "after", after2)
```